In [2]:
!apt install tesseract-ocr libtesseract-dev
!pip install -q -U google-generativeai chromadb pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following additional packages will be installed:
  libarchive-dev libleptonica-dev
The following NEW packages will be installed:
  libarchive-dev libleptonica-dev libtesseract-dev
0 upgraded, 3 newly installed, 0 to remove and 2 not upgraded.
Need to get 581 kB/3,743 kB of archives.
After this operation, 16.0 MB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1.5
Err:1 http://security.ubuntu.com/ubuntu jammy-updates/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1.5
  404  Not Found [IP: 91.189.92.22 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/liba/libarchive/libarchive-dev_3.6.0-1ubuntu1.5_amd64.deb  404  Not Found [IP: 91.189.92.22 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-mis

In [3]:
# Import necessary libraries
import time
from tqdm import tqdm
import pathlib
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import google.generativeai as genai
import pandas as pd
from PIL import Image
import pytesseract
from IPython.display import Markdown

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [1]:
from dotenv import load_dotenv
import os

# Check current working directory
print(f"Current working directory: {os.getcwd()}")

# Try to find and load .env file

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY') 
genai.configure(api_key=api_key)
print(f"API Key loaded: {api_key is not None}")



Current working directory: /content


NameError: name 'genai' is not defined

## Data Preparation

Before we begin, ensure that you've uploaded the resources.zip folder and unzipped it using the following command:

In [5]:
!wget -O resources.zip "https://video.udacity-data.com/topher/2024/June/66744e79_resources/resources.zip"

--2026-04-14 16:00:53--  https://video.udacity-data.com/topher/2024/June/66744e79_resources/resources.zip
Resolving video.udacity-data.com (video.udacity-data.com)... 104.18.39.85, 172.64.148.171, 2a06:98c1:3102::ac40:94ab, ...
Connecting to video.udacity-data.com (video.udacity-data.com)|104.18.39.85|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 286142532 (273M) [application/zip]
Saving to: ‘resources.zip’

resources.zip       100%[===================>] 272.89M   109MB/s    in 2.5s    

2026-04-14 16:00:56 (109 MB/s) - ‘resources.zip’ saved [286142532/286142532]



In [6]:
!unzip resources.zip

Archive:  resources.zip
   creating: resources/
  inflating: __MACOSX/._resources    
   creating: resources/video/
  inflating: __MACOSX/resources/._video  
  inflating: resources/.DS_Store     
  inflating: __MACOSX/resources/._.DS_Store  
   creating: resources/audio/
  inflating: __MACOSX/resources/._audio  
   creating: resources/text/
  inflating: __MACOSX/resources/._text  
  inflating: resources/video/Apollo11PlaqueComparison.mov  
  inflating: __MACOSX/resources/video/._Apollo11PlaqueComparison.mov  
  inflating: resources/video/Apollo11Intro.mov  
  inflating: __MACOSX/resources/video/._Apollo11Intro.mov  
  inflating: resources/video/Apollo11MoonwalkMontage.mov  
  inflating: __MACOSX/resources/video/._Apollo11MoonwalkMontage.mov  
  inflating: resources/video/OneSmallStepCompilation.mov  
  inflating: __MACOSX/resources/video/._OneSmallStepCompilation.mov  
  inflating: resources/video/RaisingTheAmericanFlag.mov  
  inflating: __MACOSX/resources/video/._RaisingTheAmericanFl

In [7]:
data_dir = pathlib.Path("resources/")
all_file_names = [str(file) for file in data_dir.rglob("*") if file.is_file() and not file.name.startswith('.')]

In [8]:
for file_name in all_file_names:
    print(file_name)

print(len(all_file_names))

resources/audio/Apollo11OnboardAudioHighlightClip2.mp3
resources/audio/Apollo11OnboardAudioHighlightClip1.mp3
resources/audio/Apollo11OnboardAudioHighlightClip4.mp3
resources/audio/Apollo11OnboardAudioHighlightClip3.mp3
resources/audio/Apollo11OnboardAudioHighlightClip5.mp3
resources/video/Apollo11Intro.mov
resources/video/Apollo11MoonwalkMontage.mov
resources/video/RaisingTheAmericanFlag.mov
resources/video/Apollo11PlaqueComparison.mov
resources/video/BuzzDescendsCompilation.mov
resources/video/OneSmallStepCompilation.mov
resources/text/images-023.jpg
resources/text/images-333.jpg
resources/text/images-020.jpg
14


### Text

We will use Tesseract OCR (Optical Character Recognition) to extract text from images of the NASA report.

In [9]:
pytesseract.pytesseract.tesseract_cmd = (r'/usr/bin/tesseract')

In [15]:
model = ZhipuAI(api_key=api_key)

Let's create a function to take in our images of a PDF, transcribe them into text, and summarize each of them.

In [23]:
def create_text_summary():
  path = pathlib.Path("resources/text")

  text_summary_prompt = f"""You are an assistant tailored for summarizing text for retrieval.
  These summaries will be turned into vector embeddings and used to retrieve the raw text.
  Give a concise summary of the text that is well optimized for retrieval. Here is the text."""

  images = []
  text_summaries = []

  for f in path.glob("*"):
    if f.is_dir() or f.name.startswith('.'):
      continue

    image = Image.open(f)
    extracted_text = pytesseract.image_to_string(image)
    response = model.generate_content(
              extracted_text,
              text_summary_prompt
              )

    images.append(image)
    text_summaries.append(response.text)

  return images, text_summaries

In [24]:
images, text_summaries = create_text_summary()

APIReachLimitError: Error code: 429, with error text {"error":{"code":"1113","message":"余额不足或无可用资源包,请充值。"}}